In [ ]:
# Install required system and Python dependencies.
!pip install surya-ocr


In [ ]:
# Load PDF from Google Colab or use a local fallback
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf"
    print(f"Running locally, using: {pdf_path}")


Saving aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf to aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf
Uploaded: aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf


In [ ]:
# Extract the embedded text layer from the PDF using PyMuPDF (fitz).
# Each page's text is stored in a structured DataFrame.
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.8 MB/s eta 0:00:00


In [4]:
#Convert PDF to images
import fitz  # pymupdf
from pathlib import Path

pdf_file = pdf_path
img_dir = Path("pdf_pages")
img_dir.mkdir(exist_ok=True)

doc = fitz.open(pdf_file)
dpi = 200
zoom = dpi / 72

image_paths = []

for i in range(doc.page_count):
    page = doc.load_page(i)
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img_path = img_dir / f"page_{i+1:03d}.png"
    pix.save(img_path)
    image_paths.append(img_path)

doc.close()

print(f"Rendered {len(image_paths)} pages")


Rendered 68 pages


In [ ]:
# Extract Ground Truth Text from the PDF
# from the input PDF. The extracted text serves as the ground truth
import fitz  # pymupdf

gt_parts = []
doc = fitz.open(pdf_path)
for page in doc:
    gt_parts.append(page.get_text("text"))
doc.close()

groundtruth = "\n\n".join(gt_parts)

with open("groundtruth.txt", "w", encoding="utf-8") as f:
    f.write(groundtruth)

print("Saved ground truth: groundtruth.txt")


Saved ground truth: groundtruth.txt


In [ ]:
#Ladda ner Grundtruth text
from google.colab import files
files.download("groundtruth.txt")



In [ ]:
# Check for Missing and Completed Surya OCR Outputs
from pathlib import Path

surya_root = Path("surya_out")

# This verifies which pages have already been processed by the Surya OCR engine.
missing = []
present = []

for img in image_paths:  # uses rendered pages list
    page_name = img.stem  
    # expected pattern : surya_out/page_001/page_001/results.json
    expected = surya_root / page_name / page_name / "results.json"
    if expected.exists():
        present.append(page_name)
    else:
        missing.append(page_name)

print("Present:", len(present))
print("Missing:", len(missing))
print("Missing pages:", missing[:20], ("..." if len(missing) > 20 else ""))


Present: 27
Missing: 41
Missing pages: ['page_028', 'page_029', 'page_030', 'page_031', 'page_032', 'page_033', 'page_034', 'page_035', 'page_036', 'page_037', 'page_038', 'page_039', 'page_040', 'page_041', 'page_042', 'page_043', 'page_044', 'page_045', 'page_046', 'page_047'] ...


In [12]:
#Resumable OCR run for missing pages
import subprocess
from pathlib import Path

surya_root = Path("surya_out")
surya_root.mkdir(exist_ok=True)

failed = []

for page_name in missing:
    img_path = next(p for p in image_paths if p.stem == page_name)

    out_dir = surya_root / page_name
    out_dir.mkdir(parents=True, exist_ok=True)

    already = list(out_dir.rglob("results.json"))
    if already:
        print("Skip (already done):", page_name)
        continue

    print("OCR:", page_name)

    result = subprocess.run(
        ["surya_ocr", str(img_path), "--output_dir", str(out_dir)],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("FAILED:", page_name)
        print(result.stderr[:2000])  # show first 2000 chars
        failed.append(page_name)
        continue

print("\nDone. Failed pages:", failed)


OCR: page_028
OCR: page_029
OCR: page_030
OCR: page_031
OCR: page_032
OCR: page_033
OCR: page_034
OCR: page_035
OCR: page_036
OCR: page_037
OCR: page_038
OCR: page_039
OCR: page_040
OCR: page_041
OCR: page_042
OCR: page_043
OCR: page_044
OCR: page_045
OCR: page_046
OCR: page_047
OCR: page_048
OCR: page_049
OCR: page_050
OCR: page_051
OCR: page_052
OCR: page_053
OCR: page_054
OCR: page_055
OCR: page_056
OCR: page_057
OCR: page_058
OCR: page_059
OCR: page_060
OCR: page_061
OCR: page_062
OCR: page_063
OCR: page_064
OCR: page_065
OCR: page_066
OCR: page_067
OCR: page_068

Done. Failed pages: []


In [13]:
#Re-check what’s still missing
missing2 = []
present2 = []

for img in image_paths:
    page_name = img.stem
    expected = Path("surya_out") / page_name
    if list(expected.rglob("results.json")):
        present2.append(page_name)
    else:
        missing2.append(page_name)

print("Present:", len(present2))
print("Missing:", len(missing2))
print("Missing pages:", missing2)


Present: 68
Missing: 0
Missing pages: []


In [14]:
#Extract whole-PDF OCR text from all results.json
import json
from pathlib import Path

def json_to_text(data):
    if isinstance(data, dict):
        # direct text keys
        for k in ["text", "content", "ocr_text", "transcription", "prediction"]:
            v = data.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()

        # pages structure
        pages = data.get("pages")
        if isinstance(pages, list):
            parts = []
            for p in pages:
                if isinstance(p, dict):
                    if isinstance(p.get("text"), str):
                        parts.append(p["text"])
                    for ln in p.get("lines", []) if isinstance(p.get("lines"), list) else []:
                        if isinstance(ln, dict) and "text" in ln:
                            parts.append(str(ln["text"]))
                    for b in p.get("blocks", []) if isinstance(p.get("blocks"), list) else []:
                        if isinstance(b, dict) and "text" in b:
                            parts.append(str(b["text"]))
            t = "\n".join([x for x in parts if str(x).strip()]).strip()
            if t:
                return t

    # fallback: collect all "text" string fields recursively
    parts = []
    def walk(x):
        if isinstance(x, dict):
            for kk, vv in x.items():
                if kk == "text" and isinstance(vv, str):
                    parts.append(vv)
                walk(vv)
        elif isinstance(x, list):
            for vv in x:
                walk(vv)
    walk(data)
    return "\n".join([t.strip() for t in parts if t.strip()]).strip()


surya_root = Path("surya_out")

ocr_texts = []
failed_extract = []

for img in image_paths:
    page_name = img.stem
    page_dir = surya_root / page_name

    candidates = list(page_dir.rglob("results.json"))
    if not candidates:
        failed_extract.append(page_name)
        ocr_texts.append("")
        continue

    data = json.loads(candidates[0].read_text(encoding="utf-8", errors="replace"))
    text = json_to_text(data)
    if not text:
        failed_extract.append(page_name)
        ocr_texts.append("")
        continue

    ocr_texts.append(text)

print("Pages with OCR text:", sum(1 for t in ocr_texts if t.strip()))
print("Pages missing OCR text:", failed_extract)

ocr_whole_text = "\n\n".join([t for t in ocr_texts if t.strip()])
open("ocr_whole.txt", "w", encoding="utf-8").write(ocr_whole_text)

print("Saved ocr_whole.txt")


Pages with OCR text: 68
Pages missing OCR text: []
Saved ocr_whole.txt


In [21]:
#clean white spaces
import re

text = open("ocr_whole.txt", "r", encoding="utf-8", errors="replace").read()

# Remove HTML breaks
text = text.replace("<br>", "\n")

# Collapse excessive newlines
text = re.sub(r"\n{3,}", "\n\n", text)

# Normalize spaces
text = re.sub(r"[ \t]+", " ", text)




In [22]:
#Merge vertical single-letter lines
lines = text.splitlines()
fixed_lines = []

buffer = []

for line in lines:
    stripped = line.strip()

    # If line is a single character
    if len(stripped) == 1:
        buffer.append(stripped)
    else:
        if buffer:
            fixed_lines.append("".join(buffer))
            buffer = []
        fixed_lines.append(line)

# Flush buffer
if buffer:
    fixed_lines.append("".join(buffer))

clean_text = "\n".join(fixed_lines)


In [23]:
#Fixa output.md
with open("output.md", "w", encoding="utf-8") as f:
    f.write(clean_text.strip() + "\n")

print("Fixed output.md saved")


Fixed output.md saved


In [24]:
#Ladda ner
from google.colab import files
files.download("output.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>